# SolarVoix — Visualisations analytiques

**Notebook Fabric :** `NB_Visualizations`

**Lakehouse requis :** attachez `LH_SolarVoix`as the default Lakehouse before execution.

This notebook uses the table`silver_appels_enrichis`created and validated in`NB_Bronze_to_Silver`.


## Cell 1 — Imports and loading of data


In [ ]:
# === CELLULE 1 : Chargement des données pour visualisation ===
#
# On convertit la table Silver en Pandas DataFrame pour seaborn/matplotlib.
# Acceptable ici : 120 lignes. Au-delà de 100K lignes, utiliser des agrégats Spark.

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Style cohérent pour toutes les figures
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
COULEURS_SENTIMENT = {
    "positif": "#27AE60",
    "neutre":  "#F39C12",
    "negatif": "#E74C3C",
}

df = spark.table("silver_appels_enrichis").toPandas()
df["date_appel"] = pd.to_datetime(df["date_appel"])

# Convertir les entiers/booléens nullable Pandas (Int32, Int64, boolean...) en float numpy
# → compatibilité matplotlib/seaborn (ufunc isfinite)
# On exclut les StringDtype pour ne pas écraser les colonnes texte
for col in df.columns:
    if pd.api.types.is_extension_array_dtype(df[col]) and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(float)

df["semaine"] = df["date_appel"].dt.isocalendar().week.astype(float)

print(f"Données chargées : {len(df)} appels")


## Cell 2 — Scoreboard feeling and churn


In [ ]:
# === CELLULE 2 : Dashboard 4 visualisations ===
#
# VIZ 1 : Distribution des sentiments (donut chart)
# VIZ 2 : Score churn par produit (boxplot)
# VIZ 3 : Évolution temporelle des alertes (lineplot hebdomadaire)
# VIZ 4 : Heatmap heure × intention (qui appelle quand et pourquoi)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("SolarVoix — Analyse des 120 appels clients", fontsize=16, fontweight="bold", y=1.01)

# -------------------------------------------------------
# VIZ 1 : Donut — Répartition des sentiments IA
# -------------------------------------------------------
ax1 = axes[0, 0]
counts_sent = df["sentiment_ia"].value_counts()
couleurs = [COULEURS_SENTIMENT.get(s, "#95A5A6") for s in counts_sent.index]
wedges, texts, autotexts = ax1.pie(
    counts_sent.values, labels=counts_sent.index,
    autopct="%1.0f%%", colors=couleurs,
    wedgeprops={"width": 0.5},   # width < 1 → donut
    startangle=90
)
for at in autotexts:
    at.set_fontsize(11)
ax1.set_title("Répartition des sentiments (IA)", fontweight="bold")

# -------------------------------------------------------
# VIZ 2 : Boxplot — Score churn par produit
# -------------------------------------------------------
ax2 = axes[0, 1]
ordre_produits = df.groupby("produit")["score_risque_churn"].median() \
                   .sort_values(ascending=False).index
sns.boxplot(data=df, x="produit", y="score_risque_churn",
            order=ordre_produits, ax=ax2,
            palette="Reds", linewidth=1.5)
ax2.axhline(50, color="red", linestyle="--", linewidth=1, label="Seuil alerte (50)")
ax2.set_title("Score de risque churn par produit", fontweight="bold")
ax2.set_xlabel("")
ax2.set_ylabel("Score churn (0-100)")
ax2.tick_params(axis="x", rotation=20)
ax2.legend(fontsize=9)

# -------------------------------------------------------
# VIZ 3 : Lineplot — Alertes critiques par semaine
# -------------------------------------------------------
ax3 = axes[1, 0]
alertes_semaine = (
    df[df["alerte_critique"]]
    .groupby("semaine")
    .size()
    .reset_index(name="nb_alertes")
)
sns.lineplot(data=alertes_semaine, x="semaine", y="nb_alertes",
             ax=ax3, marker="o", color="#E74C3C", linewidth=2)
ax3.fill_between(alertes_semaine["semaine"], alertes_semaine["nb_alertes"],
                  alpha=0.15, color="#E74C3C")
ax3.set_title("Alertes critiques par semaine (score ≥ 50)", fontweight="bold")
ax3.set_xlabel("Semaine de l'année")
ax3.set_ylabel("Nombre d'alertes")
ax3.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# -------------------------------------------------------
# VIZ 4 : Heatmap — Intentions par tranche horaire
# -------------------------------------------------------
ax4 = axes[1, 1]
pivot = (
    df.groupby(["tranche_horaire", "intent_detecte"])
    .size()
    .unstack(fill_value=0)
)
# Garder les 6 intentions les plus fréquentes pour lisibilité
top6_intents = df["intent_detecte"].value_counts().head(6).index
pivot = pivot[[c for c in top6_intents if c in pivot.columns]]

sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=ax4, cbar_kws={"label": "Nb appels"})
ax4.set_title("Intentions par tranche horaire", fontweight="bold")
ax4.set_xlabel("")
ax4.set_ylabel("")
ax4.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("/tmp/dashboard_solarvoix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard sauvegardé")


## Cellule 3 — Matrice de performance IA


In [ ]:
# === CELLULE 3 : Évaluation de la qualité du modèle IA ===
#
# OBJECTIF : Visualiser la matrice de confusion (sentiment IA vs réel)
#            pour mesurer la fiabilité du modèle sur nos données métier.

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Filtrer les lignes avec des valeurs manquantes dans les colonnes sentiment
df_cm = df.dropna(subset=["sentiment_reel", "sentiment_ia"]).copy()

# On mappe "tres_negatif" → "negatif" pour la comparaison
# (le modèle local utilise 3 classes)
df_cm["sentiment_reel_3classes"] = df_cm["sentiment_reel"].astype(str).replace(
    {"tres_negatif": "negatif"}
)

labels = ["positif", "neutre", "negatif"]
y_true = df_cm["sentiment_reel_3classes"]
y_pred = df_cm["sentiment_ia"].astype(str)

cm = confusion_matrix(y_true, y_pred, labels=labels)
print(f"Lignes utilisées pour la matrice : {len(df_cm)} / {len(df)}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Prédiction IA", fontsize=12)
ax.set_ylabel("Vérité terrain", fontsize=12)
ax.set_title("Matrice de confusion — Modèle de sentiment", fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== RAPPORT DE CLASSIFICATION ===")
print(classification_report(y_true, y_pred, labels=labels))
